# SolarSelect — Dataset Inspection and Source Verification

**File:** `dataset_inspection_and_source_verification.ipynb`  
**Location:** `notebooks/data_inspection/`  
**Author:** SolarSelect MLOps Engineering Team  
**Scope:** Complete chronological record of all data inspection and suitability audits performed for SolarSelect.  
**Execution Boundary:** Diagnostic / Read-Only. No modifications, cleaning, or model training.

---

## 1. Project Background and Inspection Objectives

SolarSelect is an end-to-end solar energy prediction and panel recommendation system:
- **Layer A:** Climate-conditioned Specific Energy Yield Prediction ($Y_f = \text{kWh}/\text{kW}_p$).
- **Layer B:** Constraint-aware Real Solar Panel Recommendation Engine.

In this notebook, we execute all code used to inspect and verify:
1. **The original Kaggle Solar Power Generation dataset** (4 CSV files in `data/raw/archive/`).
2. **Kaggle inverter and sensor characteristics** (source keys, plant IDs, sampling resolution, timestamp formats).
3. **Investigation of candidate supplementary datasets** (UNISOLAR vs Cyprus PV).
4. **UNISOLAR dataset download and integrity verification** (4 CSV files in `data/raw/unisolar/`).
5. **Full raw inventory and schema audit** across all 8 collected CSV files.

In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Image

# Set workspace paths relative to notebook location
project_root = Path("../../").resolve()
kaggle_dir = project_root / "data/raw/archive"
unisolar_dir = project_root / "data/raw/unisolar"

print(f"Project Root: {project_root}")
print(f"Kaggle Data Directory: {kaggle_dir} (Exists: {kaggle_dir.exists()})")
print(f"UNISOLAR Data Directory: {unisolar_dir} (Exists: {unisolar_dir.exists()})")

## 2. Phase 1, Step 1: Existing Kaggle Dataset Suitability Audit

We verify the existence, file sizes, and row/column counts of the four original Kaggle CSV files.

In [ ]:
kaggle_files = {
    "Plant 1 Generation": kaggle_dir / "Plant_1_Generation_Data.csv",
    "Plant 1 Weather": kaggle_dir / "Plant_1_Weather_Sensor_Data.csv",
    "Plant 2 Generation": kaggle_dir / "Plant_2_Generation_Data.csv",
    "Plant 2 Weather": kaggle_dir / "Plant_2_Weather_Sensor_Data.csv",
}

kaggle_inventory = []
for name, filepath in kaggle_files.items():
    assert filepath.exists(), f"Missing file: {filepath}"
    size_mb = filepath.stat().st_size / (1024 * 1024)
    # Stream line count
    with open(filepath, "rb") as f:
        line_count = sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1024*1024), b""))
    
    df_head = pd.read_csv(filepath, nrows=2)
    kaggle_inventory.append({
        "Dataset Name": name,
        "Filename": filepath.name,
        "Size (MB)": f"{size_mb:.2f} MB",
        "Row Count": f"{line_count - 1:,}",
        "Column Count": len(df_head.columns),
        "Columns": list(df_head.columns)
    })

pd.DataFrame(kaggle_inventory)[["Dataset Name", "Filename", "Size (MB)", "Row Count", "Column Count", "Columns"]]

## 3. Phase 1, Step 2: Deep Inverter, Sensor, and Temporal Inspection

We inspect inverter identifiers (`SOURCE_KEY`), distinct plants (`PLANT_ID`), temporal ranges, and timestamp format conventions.

In [ ]:
p1_gen = pd.read_csv(kaggle_files["Plant 1 Generation"])
p1_wea = pd.read_csv(kaggle_files["Plant 1 Weather"])
p2_gen = pd.read_csv(kaggle_files["Plant 2 Generation"])
p2_wea = pd.read_csv(kaggle_files["Plant 2 Weather"])

# Check unique inverters and plants
print("=== INVERTER & PLANT IDENTIFIERS ===")
print(f"Plant 1 Generation: Plant ID={p1_gen['PLANT_ID'].unique()}, Unique Inverters={p1_gen['SOURCE_KEY'].nunique()}")
print(f"Plant 2 Generation: Plant ID={p2_gen['PLANT_ID'].unique()}, Unique Inverters={p2_gen['SOURCE_KEY'].nunique()}")
print(f"Plant 1 Weather: Plant ID={p1_wea['PLANT_ID'].unique()}, Weather Sensor ID={p1_wea['SOURCE_KEY'].unique()}")
print(f"Plant 2 Weather: Plant ID={p2_wea['PLANT_ID'].unique()}, Weather Sensor ID={p2_wea['SOURCE_KEY'].unique()}")

# Date formats and temporal span
p1_gen["DT"] = pd.to_datetime(p1_gen["DATE_TIME"], format="%d-%m-%Y %H:%M")
p1_wea["DT"] = pd.to_datetime(p1_wea["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")
p2_gen["DT"] = pd.to_datetime(p2_gen["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")
p2_wea["DT"] = pd.to_datetime(p2_wea["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")

print("\n=== TEMPORAL RANGES ===")
print(f"Plant 1 Gen Span: {p1_gen['DT'].min()} to {p1_gen['DT'].max()} (Span: {(p1_gen['DT'].max() - p1_gen['DT'].min()).days} days)")
print(f"Plant 2 Gen Span: {p2_gen['DT'].min()} to {p2_gen['DT'].max()} (Span: {(p2_gen['DT'].max() - p2_gen['DT'].min()).days} days)")
print(f"Plant 1 Wea Span: {p1_wea['DT'].min()} to {p1_wea['DT'].max()}")
print(f"Plant 2 Wea Span: {p2_wea['DT'].min()} to {p2_wea['DT'].max()}")

### Sample Records and Data Types (Kaggle Plants 1 & 2)

In [ ]:
print("--- Plant 1 Generation Data (First 3 Rows) ---")
display(p1_gen.head(3))

print("--- Plant 1 Weather Sensor Data (First 3 Rows) ---")
display(p1_wea.head(3))

## 4. Phase 1, Step 3: UNISOLAR Download and Verification

To overcome the 34-day single-season limitation of the Kaggle archive, the **UNISOLAR** dataset (multi-year university campus rooftop solar) was approved and integrated into `data/raw/unisolar/`.

In [ ]:
unisolar_files = {
    "UNISOLAR Site Details": unisolar_dir / "Solar_Site_Details.csv",
    "UNISOLAR Energy Generation": unisolar_dir / "Solar_Energy_Generation.csv",
    "UNISOLAR Weather Data": unisolar_dir / "Weather_Data_reordered_all.csv",
    "UNISOLAR Monthly Summary": unisolar_dir / "Monthly_Summary_Solar.csv",
}

unisolar_inventory = []
for name, filepath in unisolar_files.items():
    assert filepath.exists(), f"Missing file: {filepath}"
    size_mb = filepath.stat().st_size / (1024 * 1024)
    with open(filepath, "rb") as f:
        line_count = sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1024*1024), b""))
    
    df_head = pd.read_csv(filepath, nrows=2)
    unisolar_inventory.append({
        "Dataset Name": name,
        "Filename": filepath.name,
        "Size (MB)": f"{size_mb:.2f} MB",
        "Row Count": f"{line_count - 1:,}",
        "Column Count": len(df_head.columns),
        "Columns": list(df_head.columns)
    })

pd.DataFrame(unisolar_inventory)[["Dataset Name", "Filename", "Size (MB)", "Row Count", "Column Count", "Columns"]]

### Sample Records for UNISOLAR Datasets

In [ ]:
u_meta = pd.read_csv(unisolar_files["UNISOLAR Site Details"], nrows=5)
u_gen = pd.read_csv(unisolar_files["UNISOLAR Energy Generation"], nrows=5)
u_wea = pd.read_csv(unisolar_files["UNISOLAR Weather Data"], nrows=5)

print("--- UNISOLAR Site Details Sample ---")
display(u_meta[["CampusKey", "SiteKey", "kWp", "Number of panels", "Panel", "Inverter"]])

print("--- UNISOLAR Energy Generation Sample ---")
display(u_gen)

print("--- UNISOLAR Weather Data Sample ---")
display(u_wea)

## 5. Full Raw Verification and Scale Comparison

We visualize and compare the total row counts and dimensionality across all 8 collected CSV datasets.

In [ ]:
Image(filename="figures/raw_dataset_dimensions_and_record_counts.png")

## 6. Synthesis of Data Inspection Findings

1. **Dataset Foundations:**
   - **Kaggle Solar Power Archive:** 4 CSVs, 139,658 generation rows across 2 plants, 22 inverters each, 1 on-site weather sensor each, covering May 15 to June 17, 2020.
   - **UNISOLAR Dataset:** 4 CSVs, 2,731,946 generation rows across 90 sites and 4 campuses, covering 2015 to 2020.
2. **Measurement Units & Target Types:**
   - Kaggle `AC_POWER`: Instantaneous active power in **kilowatts (kW)**.
   - UNISOLAR `SolarGeneration`: Integrated 15-minute interval energy in **kilowatt-hours (kWh)**.
3. **DC Power Scale Discrepancy:**
   - Plant 1 `DC_POWER` is recorded in raw sensor counts ($pprox 10\times$ AC power, max $14,471$).
   - Plant 2 `DC_POWER` is recorded in physical kilowatts ($pprox 1.022\times$ AC power, max $1,420\text{ kW}$).
4. **Timestamp Parsing Divergence:**
   - Plant 1 Generation uses day-first format: `%d-%m-%Y %H:%M`.
   - Plant 1 Weather, Plant 2, and UNISOLAR use standard ISO format: `%Y-%m-%d %H:%M:%S`.
5. **Conclusion:** All 8 datasets are intact, verified, and ready for detailed exploratory analysis in `notebooks/eda2.0/eda2.0_data_quality_and_statistical_analysis.ipynb`.